# Greek Banking Sector — Advanced Financial Analysis

## DuPont Analysis & Advanced Metrics

This notebook extends the base analysis with:
- **DuPont Analysis**: ROE decomposition into profit margin, asset turnover, and financial leverage
- **NPL & Provisioning Analysis**: Asset quality metrics
- **Advanced Ratios**: NIM, cost of risk, efficiency ratios
- **Peer Comparison**: Bank-by-bank benchmarking

**Data Source**: `kpis_final.csv` (processed data from 01_extract.ipynb)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Load Data

In [ ]:
# Load KPIs dataset
df = pd.read_csv('../data/processed/kpis_final.csv')
print("📊 KPIs Dataset:")
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"\nBanks: {df['bank'].unique()}")
print(f"Years: {df['year'].unique()}")

## 1. DuPont Analysis

The DuPont system breaks down ROE into three components:

**ROE = Net Profit Margin × Asset Turnover × Financial Leverage**

- **Net Profit Margin** = Net Profit / Operating Income
- **Asset Turnover** = Operating Income / Total Assets
- **Financial Leverage** = Total Assets / Equity

In [ ]:
# DuPont Analysis
dupont = df.copy()

# Calculate DuPont components
dupont['net_profit_margin'] = (dupont['net_profit'] / dupont['operating_income'] * 100).round(2)
dupont['asset_turnover'] = (dupont['operating_income'] / dupont['total_assets'] * 100).round(2)
dupont['financial_leverage'] = (dupont['total_assets'] / dupont['equity']).round(2)

# Verify ROE calculation
dupont['roe_calculated'] = (dupont['net_profit_margin'] * dupont['asset_turnover'] * dupont['financial_leverage'] / 100).round(2)

print("📈 DuPont Analysis — 2024:")
dupont_2024 = dupont[dupont['year'] == 2024][['bank', 'net_profit_margin', 'asset_turnover', 'financial_leverage', 'roe_calculated', 'roe']]
print(dupont_2024.to_string(index=False))

In [ ]:
# Visualize DuPont components
fig = go.Figure()

banks = dupont_2024['bank'].values
x = np.arange(len(banks))
width = 0.25

fig.add_trace(go.Bar(x=x - width, y=dupont_2024['net_profit_margin'], name='Net Profit Margin (%)', marker_color='#3b82f6'))
fig.add_trace(go.Bar(x=x, y=dupont_2024['asset_turnover'], name='Asset Turnover (%)', marker_color='#10b981'))
fig.add_trace(go.Bar(x=x + width, y=dupont_2024['financial_leverage'], name='Financial Leverage (x)', marker_color='#f59e0b'))

fig.update_layout(
    title='DuPont Analysis — 2024',
    barmode='group',
    xaxis_title='Bank',
    yaxis_title='Value',
    legend_title='Component',
    template='plotly_dark'
)

fig.show()

## 2. NPL & Provisioning Analysis

Note: Full NPL data requires more granular disclosure from annual reports. Here we analyze provisioning trends.

In [ ]:
# Provisioning Analysis
fig = make_subplots(rows=1, cols=2, subplot_titles=('Impairment Losses (€m)', 'Cost of Risk (% of Loans)'))

for bank in df['bank'].unique():
    bank_data = df[df['bank'] == bank].sort_values('year')
    
    # Impairment chart
    fig.add_trace(
        go.Scatter(x=bank_data['year'], y=bank_data['impairment'].abs(), mode='lines+markers', name=bank),
        row=1, col=1
    )
    
    # Cost of Risk = Impairment / Average Loans
    bank_data = bank_data.copy()
    bank_data['cost_of_risk'] = (bank_data['impairment'].abs() / bank_data['loans'] * 100).round(2)
    
    fig.add_trace(
        go.Scatter(x=bank_data['year'], y=bank_data['cost_of_risk'], mode='lines+markers', name=bank, showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Asset Quality — Provisioning Trends', template='plotly_dark', height=400)
fig.update_xaxes(title='Year', row=1, col=1)
fig.update_xaxes(title='Year', row=1, col=2)
fig.update_yaxes(title='€ million', row=1, col=1)
fig.update_yaxes(title='% of Loans', row=1, col=2)

fig.show()

## 3. Net Interest Margin (NIM) Analysis

In [ ]:
# NIM Comparison
fig = go.Figure()

for bank in df['bank'].unique():
    bank_data = df[df['bank'] == bank].sort_values('year')
    fig.add_trace(go.Bar(name=bank, x=bank_data['year'], y=bank_data['nim']))

fig.update_layout(
    title='Net Interest Margin by Bank (%)',
    xaxis_title='Year',
    yaxis_title='NIM (%)',
    barmode='group',
    template='plotly_dark'
)

fig.show()

## 4. Efficiency Analysis (Cost-to-Income)

In [ ]:
# Cost-to-Income trends
fig = go.Figure()

for bank in df['bank'].unique():
    bank_data = df[df['bank'] == bank].sort_values('year')
    fig.add_trace(go.Scatter(x=bank_data['year'], y=bank_data['cost_to_income'], mode='lines+markers', name=bank))

fig.update_layout(
    title='Cost-to-Income Ratio (%) — Lower is Better',
    xaxis_title='Year',
    yaxis_title='Cost-to-Income (%)',
    template='plotly_dark'
)

fig.show()

## 5. Peer Comparison Heatmap

In [ ]:
# Create comparison matrix for 2024
df_2024 = df[df['year'] == 2024].set_index('bank')

metrics = ['roe', 'nim', 'cost_to_income', 'loan_to_deposit']
metric_labels = {'roe': 'ROE (%)', 'nim': 'NIM (%)', 'cost_to_income': 'Cost-to-Income (%)', 'loan_to_deposit': 'Loan-to-Deposit (%)'}

heatmap_data = df_2024[metrics].T
heatmap_data.index = [metric_labels[m] for m in metrics]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdYlGn',
    text=heatmap_data.values,
    texttemplate='%{z:.1f}',
    textfont={'size': 14}
))

fig.update_layout(title='Bank Comparison Heatmap — 2024', template='plotly_dark', height=350)
fig.show()

## 6. Summary Statistics

In [ ]:
# Summary table
summary = df_2024[['nii', 'operating_income', 'net_profit', 'roe', 'nim', 'cost_to_income']].round(1)
summary.columns = ['NII (€m)', 'Op. Income (€m)', 'Net Profit (€m)', 'ROE (%)', 'NIM (%)', 'Cost/Income (%)']

print("📊 2024 Summary:")
print(summary.to_string())

---

*Analysis completed. See DATA_DICTIONARY.md for metric definitions.*